In [63]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [64]:
!pip install --upgrade opencv-python opencv-contrib-python
!pip install ultralytics paddleocr opencv-python-headless
!pip install paddleocr
!pip install paddlepaddle
!pip install ultralytics

In [68]:
import cv2
import os
import numpy as np
from ultralytics import YOLO
from paddleocr import PaddleOCR

class YOLOPaddleOCRPipeline:
    def __init__(self, model_path, classnames, ocr_language='en'):
        self.model = YOLO(model_path)
        self.classnames = classnames
        self.ocr = PaddleOCR(use_angle_cls=True, lang=ocr_language)

    @staticmethod
    def preprocess_image(image):
        def bilateral_filter(image, d=7, sigma_color=50, sigma_space=50):
            return cv2.bilateralFilter(image, d, sigma_color, sigma_space)

        def balanced_sharpen_image(image):
            kernel = np.array([[0, -0.5, 0],
                               [-0.5, 4, -0.5],
                               [0, -0.5, 0]])
            return cv2.filter2D(image, -1, kernel)

        def increase_saturation(image, saturation_scale=1.2):
            hsv = cv2.cvtColor(image, cv2.COLOR_BGR2HSV)
            hsv[:, :, 1] = cv2.multiply(hsv[:, :, 1], saturation_scale)
            return cv2.cvtColor(hsv, cv2.COLOR_HSV2BGR)

        filtered_image = bilateral_filter(image)
        sharpened_image = balanced_sharpen_image(filtered_image)
        saturated_image = increase_saturation(sharpened_image)
        return saturated_image

    def perform_ocr(self, image):
        result = self.ocr.ocr(image, cls=True)
        # Check if result[0] is valid before iterating
        if result and result[0]:
            extracted_text = " ".join([line[1][0] for line in result[0]])
            return extracted_text.strip()
        else:
            return ""  # Return an empty string if no text is detected

    def process_images(self, dataset_paths, output_folder):
        os.makedirs(output_folder, exist_ok=True)

        # Menggabungkan semua file gambar dari folder dataset
        file_list = []
        for path in dataset_paths:
            if os.path.exists(path):
                file_list += [os.path.join(path, f) for f in os.listdir(path) if f.lower().endswith(('.jpg', '.jpeg', '.png'))]

        if not file_list:
            print("Tidak ada gambar dalam folder.")
            return

        for file_path in file_list:
            frame = cv2.imread(file_path)
            if frame is None:
                print(f"Gagal membaca file: {file_path}")
                continue

            frame = cv2.resize(frame, (1080, 720))
            preprocessed_frame = self.preprocess_image(frame)

            results = self.model(preprocessed_frame)
            base_name = os.path.splitext(os.path.basename(file_path))[0]
            crop_output_folder = os.path.join(output_folder, base_name)
            os.makedirs(crop_output_folder, exist_ok=True)

            crop_count = 0
            for result in results:
                for box in result.boxes:
                    x1, y1, x2, y2 = box.xyxy[0]
                    x1, y1, x2, y2 = int(x1), int(y1), int(x2), int(y2)
                    confidence = box.conf[0]
                    class_detect = int(box.cls[0])
                    class_name = self.classnames[class_detect]
                    conf = int(confidence * 100)

                    if conf > 30 and class_name == 'license-plate':
                        cropped_image = frame[y1:y2, x1:x2]
                        ocr_result = self.perform_ocr(cropped_image)

                        crop_filename = os.path.join(
                            crop_output_folder, f"crop_{crop_count}_{ocr_result or 'unread'}.jpg"
                        )
                        cv2.imwrite(crop_filename, cropped_image)
                        crop_count += 1

            output_filename = os.path.join(output_folder, os.path.basename(file_path))
            cv2.imwrite(output_filename, frame)
            print(f"Hasil disimpan: {output_filename}, {crop_count} hasil crop disimpan di {crop_output_folder}")


In [66]:
dataset_path = ['/content/drive/My Drive/Colab Notebooks/comvis_platnomor/3.TL.Kartini']  # Folder dataset yang berisi gambar dan file annotasi
output_folder = '/content/drive/My Drive/Colab Notebooks/comvis_platnomor/output_folder'  # Folder output hasil crop
model_path = '/content/drive/My Drive/Colab Notebooks/comvis_platnomor/license-plate-comvis/best.pt'  # Path ke model YOLO
classnames = ['license-plate', 'vehicle']  # Kelas yang terdeteksi oleh model
class_id_target = 0  # Class ID untuk objek yang ingin dicrop

In [69]:
pipeline = YOLOPaddleOCRPipeline(model_path, classnames)
pipeline.process_images(dataset_paths, output_folder)

print("Proses selesai. Hasil tersimpan di Google Drive.")

[2025/01/09 16:51:44] ppocr DEBUG: Namespace(help='==SUPPRESS==', use_gpu=False, use_xpu=False, use_npu=False, use_mlu=False, ir_optim=True, use_tensorrt=False, min_subgraph_size=15, precision='fp32', gpu_mem=500, gpu_id=0, image_dir=None, page_num=0, det_algorithm='DB', det_model_dir='/root/.paddleocr/whl/det/en/en_PP-OCRv3_det_infer', det_limit_side_len=960, det_limit_type='max', det_box_type='quad', det_db_thresh=0.3, det_db_box_thresh=0.6, det_db_unclip_ratio=1.5, max_batch_size=10, use_dilation=False, det_db_score_mode='fast', det_east_score_thresh=0.8, det_east_cover_thresh=0.1, det_east_nms_thresh=0.2, det_sast_score_thresh=0.5, det_sast_nms_thresh=0.2, det_pse_thresh=0, det_pse_box_thresh=0.85, det_pse_min_area=16, det_pse_scale=1, scales=[8, 16, 32], alpha=1.0, beta=1.0, fourier_degree=5, rec_algorithm='SVTR_LCNet', rec_model_dir='/root/.paddleocr/whl/rec/en/en_PP-OCRv4_rec_infer', rec_image_inverse=True, rec_image_shape='3, 48, 320', rec_batch_num=6, max_text_length=25, rec_c